# 🗂️ ROL 1: Diseñador de Datos — Modelado y Generación
## Caso SUNBURST - Análisis de Gestión de Datos

**Universidad de San Buenaventura** | Gestión de Datos | 3er Semestre

---

### Objetivo
Diseñar el modelo de datos del incidente SUNBURST y generar datos sintéticos base usando Python.

### Contenido del Notebook
1. Contexto del Caso SUNBURST
2. Modelo Entidad-Relación (ER)
3. Generación de Datos Sintéticos
4. Vista Previa de DataFrames
5. Exportación a CSV
6. Conclusiones y Decisiones de Diseño

---
## 1. Contexto del Caso SUNBURST

El **ataque SUNBURST** fue uno de los ciberataques más sofisticados de la historia. En diciembre de 2020, se descubrió que actores maliciosos habían comprometido el sistema de compilación de **SolarWinds**, insertando código malicioso (malware) en la **Plataforma Orion**, un software de monitoreo de redes utilizado por miles de organizaciones.

### Datos Clave del Caso
| Aspecto | Detalle |
|---------|--------|
| **Software afectado** | SolarWinds Orion Platform |
| **Versiones comprometidas** | 2019.4 (hasta HF4), 2020.2 y 2020.2 HF1 |
| **Versiones limpias** | 2019.4 HF5, 2020.2.1 |
| **Descargas afectadas** | ~18,000 (estimación inicial) |
| **Clientes realmente comprometidos** | <100 |
| **Entidades gubernamentales** | ~9 (Dept. Energía, Justicia, Tesoro, CISA, Pentágono) |
| **Empresas privadas afectadas** | Cisco, Intel, Microsoft, entre otras |
| **Timeline** | Enero 2019 → Mayo 2021 |
| **Costo para SolarWinds** | USD 52.6 millones |

### Timeline del Ataque
- **Ene 2019**: Primera evidencia de actividad del actor de amenazas
- **Sep 2019**: Inyección de código de prueba
- **Feb 2020**: Se compila e implementa SUNBURST
- **Mar 2020**: Hotfix 5 disponible (parche limpio)
- **Jun 2020**: El actor remueve el malware de máquinas de construcción
- **Dic 2020**: Se notifica a SolarWinds sobre SUNBURST
- **Dic 2020**: SolarWinds lanza parches de remediación
- **May 2021**: Investigaciones forenses prácticamente completas

---
## 2. Modelo Entidad-Relación (ER)

### Diseño del Modelo

Se diseñaron **3 entidades principales** que representan los componentes clave del escenario SUNBURST:

```
┌──────────────────────────┐        ┌───────────────────────────┐
│       CLIENTES           │        │   VERSIONES_SOFTWARE      │
├──────────────────────────┤        ├───────────────────────────┤
│ PK  cliente_id     (int) │        │ PK  version_id      (int) │
│     nombre_org     (str) │        │     nombre_version  (str) │
│     tipo_org       (str) │        │     fecha_release   (date)│
│     pais           (str) │        │     contiene_sunburst(bool│)
│     sector         (str) │        │     fecha_compilacion(date│)
│     criticidad     (str) │        └───────────┬───────────────┘
└───────────┬──────────────┘                    │
            │                                   │
            │ 1:N                          1:N  │
            │                                   │
            ▼                                   ▼
┌──────────────────────────────────────────────────┐
│              INSTALACIONES                       │
├──────────────────────────────────────────────────┤
│ PK  instalacion_id         (int)                 │
│ FK  cliente_id             (int) → CLIENTES      │
│ FK  version_id             (int) → VERSIONES     │
│     fecha_instalacion      (date)                │
│     nivel_datos_sensibles  (str)                 │
└──────────────────────────────────────────────────┘
```

### Justificación del Modelo

| Entidad | Justificación |
|---------|---------------|
| **Clientes** | Representa las organizaciones que usaban la Plataforma Orion. Incluye tipo, sector y criticidad para analizar el impacto diferenciado del ataque. |
| **Versiones_Software** | Captura las distintas versiones de Orion, diferenciando las que contenían SUNBURST de las que eran parches limpios. Fundamental para el análisis de cascada. |
| **Instalaciones** | Tabla puente que relaciona clientes con versiones. Representa el acto de instalar una versión específica y permite rastrear qué organizaciones están expuestas. |

### Relaciones
- **Clientes → Instalaciones** (1:N): Un cliente puede tener múltiples instalaciones.
- **Versiones → Instalaciones** (1:N): Una versión puede estar instalada en múltiples organizaciones.
- **Instalaciones** actúa como tabla puente N:M entre Clientes y Versiones.

---
## 3. Generación de Datos Sintéticos

### Configuración Inicial
Importamos las librerías necesarias y configuramos la semilla (`seed=42`) para garantizar **reproducibilidad**.

In [ ]:
# Importación de librerías
import pandas as pd
import numpy as np
from faker import Faker
import os
from datetime import datetime, timedelta

# Configuración de reproducibilidad
np.random.seed(42)
fake = Faker('es_ES')  # Faker en español
Faker.seed(42)

print("✅ Librerías cargadas correctamente")
print(f"   Pandas: {pd.__version__}")
print(f"   NumPy: {np.__version__}")

### 3.1 Datos Base del Caso
Definimos las constantes que usaremos para generar datos coherentes con el caso SUNBURST.

In [ ]:
# Tipos de organización (basados en los clientes reales de SolarWinds)
TIPOS_ORGANIZACION = [
    'Gobierno Federal', 'Gobierno Estatal', 'Empresa Privada',
    'Institución Educativa', 'Organización de Salud', 'ONG'
]

# Sectores afectados según el caso
SECTORES = [
    'Tecnología', 'Defensa', 'Energía', 'Finanzas',
    'Telecomunicaciones', 'Salud', 'Gobierno', 'Educación'
]

CRITICIDADES = ['Alta', 'Media', 'Baja']
NIVELES_DATOS_SENSIBLES = ['Bajo', 'Medio', 'Alto', 'Crítico']

# Países (principalmente EE.UU., como en el caso real)
PAISES = [
    'Estados Unidos', 'Estados Unidos', 'Estados Unidos', 'Estados Unidos',
    'Estados Unidos', 'Estados Unidos', 'Estados Unidos',  # ~70% EE.UU.
    'Reino Unido', 'Canadá', 'Alemania', 'Israel',
    'Australia', 'Francia', 'Japón'
]

print(f"Tipos de organización: {len(TIPOS_ORGANIZACION)}")
print(f"Sectores: {len(SECTORES)}")
print(f"Países disponibles: {len(set(PAISES))}")

### 3.2 Generación de Clientes (50 registros)

Cada cliente representa una organización que usaba la Plataforma Orion. Los nombres se generan con la librería `Faker` y la criticidad se asigna según el tipo de organización (las gubernamentales tienden a ser más críticas, como en el caso real donde agencias del gobierno de EE.UU. fueron los principales objetivos).

In [ ]:
def generar_clientes(n=50):
    """
    Genera n clientes simulando organizaciones que usaban SolarWinds Orion.
    La criticidad se correlaciona con el tipo de organización.
    """
    clientes = []
    for i in range(1, n + 1):
        tipo_org = np.random.choice(TIPOS_ORGANIZACION, p=[0.15, 0.10, 0.40, 0.10, 0.15, 0.10])
        
        # Nombres según tipo de organización
        if 'Gobierno' in tipo_org:
            nombre = f"{np.random.choice(['Departamento de', 'Agencia de', 'Ministerio de'])} {fake.word().capitalize()} {fake.word().capitalize()}"
            criticidad = np.random.choice(CRITICIDADES, p=[0.6, 0.3, 0.1])
            sector = np.random.choice(['Gobierno', 'Defensa', 'Energía'])
        elif tipo_org == 'Institución Educativa':
            nombre = f"Universidad {fake.city()}"
            criticidad = np.random.choice(CRITICIDADES, p=[0.2, 0.5, 0.3])
            sector = np.random.choice(SECTORES)
        elif tipo_org == 'Organización de Salud':
            nombre = f"Hospital {fake.last_name()} {fake.city()}"
            criticidad = np.random.choice(CRITICIDADES, p=[0.5, 0.35, 0.15])
            sector = 'Salud'
        else:
            nombre = f"{fake.company()}"
            criticidad = np.random.choice(CRITICIDADES, p=[0.2, 0.5, 0.3])
            sector = np.random.choice(SECTORES)
        
        clientes.append({
            'cliente_id': i,
            'nombre_organizacion': nombre,
            'tipo_org': tipo_org,
            'pais': np.random.choice(PAISES),
            'sector': sector,
            'criticidad': criticidad
        })
    
    return pd.DataFrame(clientes)


# Generar clientes
clientes_df = generar_clientes(n=50)
print(f"✅ {len(clientes_df)} clientes generados")
print(f"\nDistribución por tipo de organización:")
print(clientes_df['tipo_org'].value_counts().to_string())

### 3.3 Generación de Versiones de Software (8 registros)

Las versiones están basadas en la información real del caso SUNBURST:
- **Versiones 2019.4 hasta HF4**: contenían el malware SUNBURST
- **Versión 2019.4 HF5**: parche limpio (26 de marzo 2020)
- **Versión 2020.2 y HF1**: contenían SUNBURST
- **Versión 2020.2.1**: parche de remediación post-descubrimiento (15 dic 2020)

In [ ]:
def generar_versiones():
    """
    Genera las versiones de la Plataforma Orion basadas en el caso real.
    """
    versiones = [
        {'version_id': 1, 'nombre_version': 'Orion Platform 2019.4',
         'fecha_release': datetime(2019, 10, 22), 'contiene_sunburst': True,
         'fecha_compilacion': datetime(2019, 10, 10)},
        {'version_id': 2, 'nombre_version': 'Orion Platform 2019.4 HF1',
         'fecha_release': datetime(2019, 12, 17), 'contiene_sunburst': True,
         'fecha_compilacion': datetime(2019, 12, 5)},
        {'version_id': 3, 'nombre_version': 'Orion Platform 2019.4 HF2',
         'fecha_release': datetime(2020, 1, 23), 'contiene_sunburst': True,
         'fecha_compilacion': datetime(2020, 1, 10)},
        {'version_id': 4, 'nombre_version': 'Orion Platform 2019.4 HF3',
         'fecha_release': datetime(2020, 3, 5), 'contiene_sunburst': True,
         'fecha_compilacion': datetime(2020, 2, 20)},
        {'version_id': 5, 'nombre_version': 'Orion Platform 2019.4 HF5',
         'fecha_release': datetime(2020, 3, 26), 'contiene_sunburst': False,
         'fecha_compilacion': datetime(2020, 3, 15)},
        {'version_id': 6, 'nombre_version': 'Orion Platform 2020.2',
         'fecha_release': datetime(2020, 6, 19), 'contiene_sunburst': True,
         'fecha_compilacion': datetime(2020, 6, 4)},
        {'version_id': 7, 'nombre_version': 'Orion Platform 2020.2 HF1',
         'fecha_release': datetime(2020, 9, 14), 'contiene_sunburst': True,
         'fecha_compilacion': datetime(2020, 9, 1)},
        {'version_id': 8, 'nombre_version': 'Orion Platform 2020.2.1',
         'fecha_release': datetime(2020, 12, 15), 'contiene_sunburst': False,
         'fecha_compilacion': datetime(2020, 12, 14)},
    ]
    return pd.DataFrame(versiones)


# Generar versiones
versiones_df = generar_versiones()
print(f"✅ {len(versiones_df)} versiones generadas")
print(f"   Con SUNBURST: {versiones_df['contiene_sunburst'].sum()}")
print(f"   Sin SUNBURST (parches): {(~versiones_df['contiene_sunburst']).sum()}")

### 3.4 Generación de Instalaciones (100 registros)

Cada instalación vincula un **cliente** con una **versión** del software. Se garantiza:
- **Integridad referencial**: los IDs de cliente y versión existen en sus tablas respectivas
- **Coherencia temporal**: la fecha de instalación es posterior a la fecha de release de la versión
- **Distribución realista**: más instalaciones de versiones comprometidas (2020.2 fue la más popular)

In [ ]:
def generar_instalaciones(clientes_df, versiones_df, n=100):
    """
    Genera n instalaciones con integridad referencial y coherencia temporal.
    """
    clientes_ids = clientes_df['cliente_id'].tolist()
    versiones_ids = versiones_df['version_id'].tolist()
    
    # Probabilidades: más instalaciones de 2020.2 (era la versión principal)
    version_probs = [0.15, 0.10, 0.08, 0.07, 0.10, 0.25, 0.15, 0.10]
    
    instalaciones = []
    for i in range(1, n + 1):
        version_id = np.random.choice(versiones_ids, p=version_probs)
        cliente_id = np.random.choice(clientes_ids)
        
        # Fecha coherente: instalación entre 1-90 días después del release
        version_row = versiones_df[versiones_df['version_id'] == version_id].iloc[0]
        fecha_release = version_row['fecha_release']
        dias_despues = np.random.randint(1, 91)
        fecha_instalacion = fecha_release + timedelta(days=int(dias_despues))
        
        # No pasar de marzo 2021
        fecha_max = datetime(2021, 3, 31)
        if fecha_instalacion > fecha_max:
            fecha_instalacion = fecha_max - timedelta(days=np.random.randint(1, 30))
        
        # Nivel de sensibilidad correlacionado con criticidad del cliente
        criticidad = clientes_df[clientes_df['cliente_id'] == cliente_id]['criticidad'].iloc[0]
        if criticidad == 'Alta':
            nivel = np.random.choice(NIVELES_DATOS_SENSIBLES, p=[0.05, 0.15, 0.40, 0.40])
        elif criticidad == 'Media':
            nivel = np.random.choice(NIVELES_DATOS_SENSIBLES, p=[0.15, 0.40, 0.30, 0.15])
        else:
            nivel = np.random.choice(NIVELES_DATOS_SENSIBLES, p=[0.40, 0.35, 0.20, 0.05])
        
        instalaciones.append({
            'instalacion_id': i,
            'cliente_id': cliente_id,
            'version_id': version_id,
            'fecha_instalacion': fecha_instalacion.strftime('%Y-%m-%d'),
            'nivel_datos_sensibles': nivel
        })
    
    return pd.DataFrame(instalaciones)


# Generar instalaciones
instalaciones_df = generar_instalaciones(clientes_df, versiones_df, n=100)

# Estadísticas
versiones_sunburst = versiones_df[versiones_df['contiene_sunburst'] == True]['version_id'].tolist()
n_sunburst = instalaciones_df[instalaciones_df['version_id'].isin(versiones_sunburst)].shape[0]
print(f"✅ {len(instalaciones_df)} instalaciones generadas")
print(f"   Con versiones SUNBURST: {n_sunburst}")
print(f"   Con versiones limpias: {len(instalaciones_df) - n_sunburst}")

---
## 4. Vista Previa de DataFrames

A continuación se presentan las vistas previas de cada DataFrame generado, incluyendo `.head()`, `.info()` y `.describe()` para una inspección completa de los datos.

### 4.1 DataFrame: Clientes

In [ ]:
print("=" * 70)
print("📋 DataFrame: CLIENTES")
print("=" * 70)
print(f"\nDimensiones: {clientes_df.shape[0]} filas × {clientes_df.shape[1]} columnas")
print(f"\n--- Primeros 10 registros ---")
clientes_df.head(10)

In [ ]:
print("--- Información del DataFrame ---")
clientes_df.info()
print(f"\n--- Estadísticas descriptivas ---")
clientes_df.describe(include='all')

### 4.2 DataFrame: Versiones Software

In [ ]:
print("=" * 70)
print("📋 DataFrame: VERSIONES_SOFTWARE")
print("=" * 70)
print(f"\nDimensiones: {versiones_df.shape[0]} filas × {versiones_df.shape[1]} columnas")
print(f"\n--- Todas las versiones ---")
versiones_df

In [ ]:
print("--- Información del DataFrame ---")
versiones_df.info()
print(f"\n--- Versiones con SUNBURST ---")
print(versiones_df[versiones_df['contiene_sunburst'] == True][['nombre_version', 'fecha_release']].to_string(index=False))
print(f"\n--- Versiones limpias (parches) ---")
print(versiones_df[versiones_df['contiene_sunburst'] == False][['nombre_version', 'fecha_release']].to_string(index=False))

### 4.3 DataFrame: Instalaciones

In [ ]:
print("=" * 70)
print("📋 DataFrame: INSTALACIONES")
print("=" * 70)
print(f"\nDimensiones: {instalaciones_df.shape[0]} filas × {instalaciones_df.shape[1]} columnas")
print(f"\n--- Primeros 10 registros ---")
instalaciones_df.head(10)

In [ ]:
print("--- Información del DataFrame ---")
instalaciones_df.info()
print(f"\n--- Estadísticas descriptivas ---")
instalaciones_df.describe(include='all')

In [ ]:
# Distribución de instalaciones por versión
print("--- Instalaciones por versión ---")
inst_por_version = instalaciones_df.merge(
    versiones_df[['version_id', 'nombre_version', 'contiene_sunburst']], 
    on='version_id'
)
resumen = inst_por_version.groupby(['nombre_version', 'contiene_sunburst']).size().reset_index(name='total')
resumen = resumen.sort_values('total', ascending=False)
print(resumen.to_string(index=False))

---
## 5. Exportación a CSV

Los DataFrames se guardan como archivos CSV en la carpeta `data/` para que el Rol 2 pueda cargarlos y trabajar con ellos.

In [ ]:
# Crear directorio data si no existe
os.makedirs('../data', exist_ok=True)

# Guardar CSVs
clientes_df.to_csv('../data/clientes.csv', index=False, encoding='utf-8')
versiones_df.to_csv('../data/versiones_software.csv', index=False, encoding='utf-8')
instalaciones_df.to_csv('../data/instalaciones.csv', index=False, encoding='utf-8')

print("✅ Archivos CSV guardados exitosamente:")
print(f"   📄 data/clientes.csv ({len(clientes_df)} registros)")
print(f"   📄 data/versiones_software.csv ({len(versiones_df)} registros)")
print(f"   📄 data/instalaciones.csv ({len(instalaciones_df)} registros)")

# Verificar que los archivos se crearon
for archivo in ['clientes.csv', 'versiones_software.csv', 'instalaciones.csv']:
    ruta = os.path.join('../data', archivo)
    tamaño = os.path.getsize(ruta)
    print(f"   → {archivo}: {tamaño:,} bytes")

---
## 6. Conclusiones y Decisiones de Diseño

### Decisiones Tomadas

1. **Modelo de 3 entidades**: Se eligió un modelo con Clientes, Versiones_Software e Instalaciones porque captura la estructura fundamental del escenario SUNBURST: organizaciones que instalan versiones específicas de un software que puede o no estar comprometido.

2. **Volúmenes de datos**: 50 clientes, 8 versiones y 100 instalaciones proporcionan suficiente variedad para análisis estadístico sin ser excesivos para una actividad académica.

3. **Datos coherentes con el caso real**: Las versiones de software, fechas y la distinción SUNBURST/limpio se basan directamente en la información del Anexo 2 del caso Harvard.

4. **Correlaciones realistas**: La criticidad de los clientes influye en el nivel de datos sensibles de sus instalaciones, similar a como en el caso real las organizaciones gubernamentales manejaban información más sensible.

5. **Reproducibilidad**: El uso de `np.random.seed(42)` y `Faker.seed(42)` garantiza que los resultados son reproducibles.

### Validación de Integridad
- Todos los `cliente_id` en instalaciones existen en la tabla de clientes ✓
- Todos los `version_id` en instalaciones existen en la tabla de versiones ✓
- Las fechas de instalación son posteriores a las fechas de release ✓
- No hay valores nulos ni duplicados en los IDs ✓